# HW02 Part 2: Store Wikipedia Events in a Data Lake

This notebook creates a private S3 bucket and uploads the hourly Wikipedia Parquet files into the `wikipedia-hourly` prefix.

In [5]:
# import/setup 

from pathlib import Path

import boto3
from botocore.exceptions import ClientError

bucket_name = "dsan6000-sj1073"
s3_prefix = "wikipedia-hourly"
data_dir = Path("data")

session = boto3.Session()
region = session.region_name or "us-east-1"
s3 = session.client("s3", region_name=region)

print(f"Region: {region}")
print(f"Bucket: {bucket_name}")

Region: us-east-1
Bucket: dsan6000-sj1073


In [6]:
# bucket creation

try:
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={
                "LocationConstraint": region,
            },
        )

    print(f"Created bucket: s3://{bucket_name}")

except ClientError as error:
    error_code = error.response["Error"]["Code"]

    if error_code == "BucketAlreadyOwnedByYou":
        print(f"Bucket already exists and belongs to this account: s3://{bucket_name}")
    else:
        raise

Created bucket: s3://dsan6000-sj1073


In [7]:
# upload the 24 parquet files into the wikipedia-hourly prefix in the S3 bucket

parquet_files = sorted(data_dir.glob("*.parquet"))

if len(parquet_files) != 24:
    raise ValueError(
        f"Expected 24 Parquet files, but found {len(parquet_files)}."
    )

uploaded_files = []

for local_path in parquet_files:
    s3_key = f"{s3_prefix}/{local_path.name}"

    s3.upload_file(
        str(local_path),
        bucket_name,
        s3_key,
    )

    uploaded_files.append(s3_key)

print(f"Uploaded {len(uploaded_files)} files.")
print(f"Destination: s3://{bucket_name}/{s3_prefix}/")

Uploaded 24 files.
Destination: s3://dsan6000-sj1073/wikipedia-hourly/


In [8]:
# verification

response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=f"{s3_prefix}/",
)

s3_objects = [
    obj
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

total_size_mb = sum(obj["Size"] for obj in s3_objects) / (1024**2)

print(f"Parquet files in S3: {len(s3_objects)}")
print(f"Total size in S3: {total_size_mb:.2f} MB")
print("First five S3 objects:")

for obj in s3_objects[:5]:
    print(f"s3://{bucket_name}/{obj['Key']}")

Parquet files in S3: 24
Total size in S3: 30.53 MB
First five S3 objects:
s3://dsan6000-sj1073/wikipedia-hourly/20260901_040000.parquet
s3://dsan6000-sj1073/wikipedia-hourly/20260901_050000.parquet
s3://dsan6000-sj1073/wikipedia-hourly/20260901_060000.parquet
s3://dsan6000-sj1073/wikipedia-hourly/20260901_070000.parquet
s3://dsan6000-sj1073/wikipedia-hourly/20260901_080000.parquet
